# IEEE-CIS Fraud Detection: Memory-Aware Baseline

This experiment joins transaction and identity data, applies a stratified split, encodes categorical features consistently, and reports validation ROC-AUC. Paths and model settings are read from `config/config.yaml`.

In [ ]:
from pathlib import Path
import sys
import yaml
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'config' / 'config.yaml').exists():
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
with (PROJECT_ROOT / 'config' / 'config.yaml').open() as config_file:
    CONFIG = yaml.safe_load(config_file)
SEED = CONFIG['project']['random_seed']
RAW_DIR = PROJECT_ROOT / CONFIG['data']['raw_dir']
print(f'Project root: {PROJECT_ROOT}')
print(f'Raw data directory: {RAW_DIR}')

In [ ]:
def reduce_memory_usage(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Downcast numeric columns while retaining missing values."""
    for column in dataframe.columns:
        if pd.api.types.is_float_dtype(dataframe[column].dtype):
            dataframe[column] = pd.to_numeric(dataframe[column], downcast='float')
        elif pd.api.types.is_integer_dtype(dataframe[column].dtype):
            dataframe[column] = pd.to_numeric(dataframe[column], downcast='integer')
    return dataframe

def load_training_data() -> pd.DataFrame:
    """Load and validate the transaction/identity one-to-one join."""
    transactions = pd.read_csv(RAW_DIR / CONFIG['data']['train_transaction'])
    identity = pd.read_csv(RAW_DIR / CONFIG['data']['train_identity'])
    merged = transactions.merge(identity, on=CONFIG['data']['id_column'], how='left', validate='one_to_one')
    return reduce_memory_usage(merged)

data = load_training_data()
target_column = CONFIG['data']['target_column']
print(f'Loaded shape: {data.shape}; fraud rate: {data[target_column].mean():.4%}')

## Stratified split and categorical preparation

Categorical levels are learned from train and validation values, then represented as integer codes. Missing values receive a numeric sentinel.

In [ ]:
y = data.pop(target_column).astype('int8')
data = data.drop(columns=[CONFIG['data']['id_column']], errors='ignore')
X_train, X_valid, y_train, y_valid = train_test_split(data, y, test_size=CONFIG['validation']['test_size'], stratify=y, random_state=SEED)

def encode_categoricals(train: pd.DataFrame, valid: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Apply identical category codes to train and validation partitions."""
    train_encoded, valid_encoded = train.copy(), valid.copy()
    for column in train_encoded.columns:
        if train_encoded[column].dtype == 'object':
            combined = pd.concat([train_encoded[column], valid_encoded[column]], ignore_index=True).astype('category')
            train_encoded[column] = combined.iloc[:len(train_encoded)].cat.codes.astype('int32').to_numpy()
            valid_encoded[column] = combined.iloc[len(train_encoded):].cat.codes.astype('int32').to_numpy()
    return train_encoded.fillna(-999), valid_encoded.fillna(-999)

X_train, X_valid = encode_categoricals(X_train, X_valid)
print(f'Train: {X_train.shape}; validation: {X_valid.shape}')

In [ ]:
model_config = CONFIG['model']
scale_pos_weight = float((y_train == 0).sum() / max((y_train == 1).sum(), 1))
try:
    from lightgbm import LGBMClassifier
    model = LGBMClassifier(n_estimators=model_config['n_estimators'], learning_rate=model_config['learning_rate'], num_leaves=model_config['num_leaves'], max_depth=model_config['max_depth'], subsample=model_config['subsample'], colsample_bytree=model_config['colsample_bytree'], objective='binary', scale_pos_weight=scale_pos_weight, random_state=SEED, n_jobs=model_config['n_jobs'], verbosity=-1)
    model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], callbacks=[])
except ImportError as error:
    raise ImportError('Install requirements.txt to run the LightGBM baseline.') from error
validation_probabilities = model.predict_proba(X_valid)[:, 1]
baseline_roc_auc = roc_auc_score(y_valid, validation_probabilities)
print(f'Baseline validation ROC-AUC: {baseline_roc_auc:.6f}')

## Interpretation

The printed ROC-AUC is a quick benchmark only. Before trusting improvements, compare stronger validation schemes, inspect temporal and identity distribution shift, and save the model plus preprocessing metadata together.